# TC4: Đánh đổi vector và Weighted K-means K=16 — Fashion-MNIST 60k

Notebook này hiện thực khảo sát tại `docs/UD2/TC4_Vector_PreKMeans_Tradeoff_60k.md` và benchmark tại `docs/UD2/TC/spec_weighted_kmeans_k16_timing.md`. Pha A chọn 2–3 phương án đáng thử bằng **proxy chi phí – độ méo vector – bảo toàn thứ tự khoảng cách**; pha B đo runtime Weighted K-means thực tế với K=16.

> **Phạm vi:** pha A dùng toàn bộ descriptor train sau TC0–TC3 để khảo sát 14 biểu diễn trước K-means. Pha B fit Weighted K-means K=16 trên train và dùng split validation chỉ để chấm Raw-space Validation Distortion. Validation không tham gia fit; notebook không chạy TC5, TC6 hoặc Eclat.

Pha A chạy B0, P12/P08/P04, Q05/Q10/Q20/Q32 và PQ12/PQ08 với α∈{0,10; 0,25; 0,50}. Pha B mặc định minh họa B0, Q32, PQ12_A50; bạn tự sửa danh sách trong cell cấu hình benchmark. Các biểu diễn được dựng lần lượt để giới hạn bộ nhớ.


## 1. Khởi tạo môi trường

Cell tìm repo root, kích hoạt Project.toml của repo và nạp package. Mặc định dùng cấu hình full CPU chỉ để dựng dữ liệu TC0–TC3; cell phía dưới không gọi `run_tc4!`.


In [ ]:
using Pkg
using TOML
using LinearAlgebra
using Random
using Statistics

function find_repo_root(start = pwd())
    current = abspath(start)
    while true
        isfile(joinpath(current, "Project.toml")) && return current
        parent = dirname(current)
        parent == current && error("Không tìm thấy Project.toml từ $start")
        current = parent
    end
end

const REPO_ROOT = find_repo_root()
cd(REPO_ROOT)
Pkg.activate(REPO_ROOT)
Pkg.instantiate()
using EclatFIM

const TC = EclatFIM.FashionTC
println("Repo: ", REPO_ROOT)
println("Julia: ", VERSION)
println("Project: ", Base.active_project())

## 2. Cấu hình thí nghiệm

Đường dẫn dữ liệu mặc định theo notebook TC3. Có thể đổi `DATA_DIR` nếu archive Fashion-MNIST nằm nơi khác. Output chỉ chứa report/biểu đồ; file descriptor trung gian được đặt trong thư mục tạm local.

In [ ]:
const CONFIG_PATH = joinpath(EclatFIM.FASHION_MNIST_ROOT, "configs", "transactions", "split_full.toml")
const DATA_DIR = joinpath(REPO_ROOT, "data", "raw", "fashion-mnist")
const REPORT_DIR = joinpath(REPO_ROOT, "results", "fashion", "tc", "tc4_vector_tradeoff_60k")
const PAIR_SEED = 42
const CHUNK_SIZE = 32_768
const PAIR_SAMPLE_SIZE = 30_000
const EXPECTED_N_REFERENCE = 1_836_369

mkpath(REPORT_DIR)
println("Config: ", relpath(CONFIG_PATH, REPO_ROOT))
println("Data: ", relpath(DATA_DIR, REPO_ROOT))
println("Report: ", relpath(REPORT_DIR, REPO_ROOT))
const TC4_PREKMEANS_RUN_KMEANS = false
@assert !TC4_PREKMEANS_RUN_KMEANS
println("Pha khảo sát vector không fit K-means: ", !TC4_PREKMEANS_RUN_KMEANS)


## 3. Đóng băng đầu vào train qua TC0–TC3

Các bước dưới đây dùng chính pipeline hiện có: xác minh archive và fit Q10+ trên train (TC0), xác định lưới patch 4×4/stride 4 (TC1), lọc theo `max(pixel) > tau_train` (TC2), tạo Raw Pixel 16D Float32/255 (TC3). Ở full mode TC1 không lưu toàn bộ patch object; TC3 tạo ma trận descriptor `16 × N`.

Chạy tuần tự từng cell để thấy tiến độ. Đây là lượt quét toàn bộ 60.000 ảnh và có thể mất thời gian.

In [ ]:
WORK_ROOT = mktempdir(prefix = "tc4-vector-tradeoff-work-")
OUTPUT_ROOT = joinpath(REPORT_DIR, "tc0_tc3_artifacts")
state = TC.prepare_run(CONFIG_PATH; data_dir = DATA_DIR,
    output_root = OUTPUT_ROOT, work_root = WORK_ROOT)

@assert state.config["run"]["mode"] == "full"
@assert state.config["tc1_patch_extraction"]["patch_size"] == [4, 4]
@assert state.config["tc2_patch_filtering"]["alpha"] == 0.0
println("State sẵn sàng; backend cấu hình: ", state.data[:backend])


In [ ]:
tc0_summary = TC.run_tc0!(state)
println(tc0_summary)
println("Tau train-only: ", state.data[:tau])
println("Ảnh train: ", length(state.data[:train_ids]))


In [ ]:
tc1_summary = TC.run_tc1!(state)
println(tc1_summary)
println("TC1 chỉ ghi số patch dự kiến ở full mode; không materialize danh sách patch.")


In [ ]:
tc2_summary = TC.run_tc2!(state)
println(tc2_summary)
println("Số patch giữ sau TC2: ", tc2_summary.summary.train_kept)
println("Duplicate rate raw sau TC2: ", tc2_summary.summary.train_duplicate_rate)


In [ ]:
tc3_summary = TC.run_tc3!(state)
X = state.data[:train_desc]

@assert length(state.data[:train_ids]) == 60_000
@assert length(state.data[:validation_ids]) == 10_000
@assert size(X, 1) == 16
@assert size(X, 2) == length(state.data[:train_map])
@assert eltype(X) == Float32
@assert all(isfinite, X)
@assert all(x -> 0f0 <= x <= 1f0, X)

N = size(X, 2)
println("TC3: ", tc3_summary)
println("Descriptor matrix: ", size(X), "; N thực tế = ", N)
println("Tau train = ", state.data[:tau])
println("Checksum descriptor cache = ", state.data[:descriptor_checksums]["train"])
println("Raw duplicate rate TC2 = ", state.data[:patch_duplicate_summary]["train"].duplicate_rate)
println("Mốc duplicate rate trước đó = 0.00537038035; đối chiếu với giá trị vừa tính.")
if N != EXPECTED_N_REFERENCE
    @warn "N khác mốc kế hoạch; tiếp tục dùng N thực tế và ghi rõ trong report." expected = EXPECTED_N_REFERENCE actual = N
end


## 4. Kiểm tra đầu vào và tính PCA basis một lần

`X` có chiều `16 × N`: mỗi cột là một patch descriptor. PCA được fit bằng covariance 16×16 trên toàn bộ train, không chuẩn hóa từng chiều và không whitening. Tích lũy theo chunk bằng Float64 để giảm sai số số học.

In [ ]:
pca_started = time()
mu, pca_basis, pca_eigenvalues = TC.covariance_pca_basis(X; chunk_size = CHUNK_SIZE)
PCA_FIT_SECONDS = time() - pca_started
raw_mean_feature_variance = sum(pca_eigenvalues) / size(X, 1)
pca = (; mu, pca_basis, pca_eigenvalues)
features = (; X, N, CHUNK_SIZE, pca, PCA_FIT_SECONDS, raw_mean_feature_variance)

@assert size(pca_basis) == (16, 16)
@assert all(isfinite, pca_eigenvalues)
println("PCA fit một lần: ", round(PCA_FIT_SECONDS; digits = 2), " giây")
println("Mean feature variance raw: ", raw_mean_feature_variance)

## 5. Hàm đánh giá dùng chung

Mỗi cấu hình tái tạo xấp xỉ descriptor gốc về không gian 16D. `raw_vector_mse` và P95 RMSE được tính trên mọi descriptor. Spearman dùng cùng 30.000 cặp train cố định cho mọi cấu hình, chỉ là metric phụ; không tạo ma trận khoảng cách N×N. Với phương án dedup, khóa số nguyên chính xác và counts luôn được giữ làm weights.

In [ ]:
pair_rng = MersenneTwister(PAIR_SEED)
pair_i = rand(pair_rng, 1:N, PAIR_SAMPLE_SIZE)
pair_j = rand(pair_rng, 1:N, PAIR_SAMPLE_SIZE)
for index in eachindex(pair_i)
    while pair_i[index] == pair_j[index]
        pair_j[index] = rand(pair_rng, 1:N)
    end
end
raw_pair_distances = [TC.distance_for_pair(X, pair_i[i], pair_j[i])
    for i in eachindex(pair_i)]
pair_ids = sort!(unique!(vcat(pair_i, pair_j)))
pair_slot = Dict(id => index for (index, id) in enumerate(pair_ids))

results = Dict{String,Any}[]

pair_sample = (; pair_i, pair_j, pair_ids, pair_slot, raw_pair_distances)

### 5.1. Ghi nhận cấu hình và nguyên tắc

Các phương án PCA dùng cùng basis đã fit; P04 là stress test. Q lượng tử hóa từng pixel trên thang UInt8 rồi chia sẻ vector trùng. PQ lượng tử hóa score PCA theo `alpha × σ_j`, với khóa tuple số nguyên. `q` và `alpha` thuộc hai không gian khác nhau.

In [ ]:
pca_dimensions = (12, 8, 4)
pixel_steps = (5, 10, 20, 32)
pca_quant_dimensions = (12, 8)
pca_quant_alphas = (0.10, 0.25, 0.50)

println("Số cấu hình: ", 1 + length(pca_dimensions) + length(pixel_steps) +
    length(pca_quant_dimensions) * length(pca_quant_alphas))
println("PCA fit split: train; center=true; standardize=false; whiten=false")
println("Ngưỡng sàng lọc gợi ý: normalized distortion ≤ 0.05 và compute proxy ≤ 0.75")


## 6. Đánh giá B0 và PCA thuần

PCA thuần giữ nguyên N điểm nhưng giảm số chiều trong khoảng cách. Với mỗi D, inverse-transform từ D thành phần rồi đo sai lệch trong Raw 16D. Tỷ lệ phương sai giải thích được báo riêng; nó không thay thế metric méo vector.

In [ ]:
TC.evaluate_baseline_and_pca!(results, features, pair_sample, pca_dimensions)

## 7. Đánh giá lượng tử hóa pixel Q

Mỗi pixel được đổi về UInt8, lượng tử hóa bằng `q * round(pixel/q)` rồi clamp về [0,255]. Khóa 16 pixel được pack chính xác vào UInt128; mỗi khóa lưu count để làm weight. Phần này không thay đổi ngưỡng TC2.

In [ ]:
for q in pixel_steps
    TC.evaluate_pixel_quantization!(results, features, pair_sample, q)
end

## 8. Đánh giá PCA → lượng tử hóa PQ

Với từng D∈{12,8}, score được lượng tử hóa theo bước `h_j = alpha × σ_j` của component j. Nếu độ lệch chuẩn gần 0, score bin và score khôi phục được đặt bằng 0. Integer tuple là khóa; counts được bảo toàn. Sau đó tái dựng vector về Raw 16D để dùng cùng MSE/RMSE/Spearman.

In [ ]:
for dimensions in pca_quant_dimensions, alpha in pca_quant_alphas
    TC.evaluate_pca_quantization!(results, features, pair_sample, dimensions, alpha)
end

## 9. Kiểm tra invariant và bảng kết quả

`compute_proxy=(U×D_eff)/(N×16)` là lượng công việc khoảng cách lý thuyết mỗi iteration khi K không đổi; nghịch đảo là theoretical proxy, không phải speedup đo được. `peak_ram_mib` là high-water RSS của tiến trình Julia đọc từ `/proc/self/status` sau cấu hình, vì vậy là mức đỉnh tiến trình tích lũy, không phải phép đo RAM cô lập hoàn toàn cho từng config.

In [ ]:
@assert length(results) == 14
@assert all(row -> row["n_input"] == N, results)
@assert all(row -> row["sum_weights"] == N, results)
@assert all(row -> row["passes_sanity_checks"] === true, results)
@assert all(row -> isfinite(row["compute_proxy"]) && row["compute_proxy"] > 0, results)

for row in results
    row["config_id"] in ("B0", "P12", "P08", "P04") || continue
    @assert row["n_unique"] == N
end

CSV_PATH = TC.write_results_csv(joinpath(REPORT_DIR, "tc4_vector_tradeoff_60k.csv"), results)
println("Đã ghi ", CSV_PATH)
println("Số cấu hình hợp lệ: ", count(row -> row["passes_sanity_checks"], results))

## 10. Pareto frontier và hai biểu đồ

Một phương án bị dominated nếu phương án khác có cả `compute_proxy` lẫn distortion không lớn hơn, và ít nhất một chỉ số nhỏ hơn. Hai biểu đồ dùng trục y `normalized_distortion`; biểu đồ thứ hai thay proxy bằng thời gian chuẩn bị, không phải thời gian K-means.

In [ ]:
proxy_frontier = TC.pareto_rows(results, "compute_proxy", "normalized_distortion")
FIGURE_PROXY = TC.tradeoff_svg(results, "compute_proxy", "Compute proxy (lower is better)",
    "Proxy chi phí và méo vector", joinpath(REPORT_DIR, "compute_proxy_vs_distortion.svg");
    frontier = true)
FIGURE_PREP = TC.tradeoff_svg(results, "prep_time_s", "Preparation time (seconds)",
    "Thời gian chuẩn bị và méo vector", joinpath(REPORT_DIR, "prep_time_vs_distortion.svg"))
println("Số phương án trên frontier proxy: ", length(proxy_frontier))

## 11. Shortlist và kết luận có giới hạn

Bảng đề cử ưu tiên cấu hình không bị dominated và nằm trong ngưỡng sàng lọc gợi ý nếu đủ phương án; nếu không đủ, notebook vẫn lấy các phương án Pareto có distortion thấp nhất để người đọc xem xét. Đây chưa phải kết luận cấu hình K-means tốt nhất.

In [ ]:
eligible = [row for row in proxy_frontier if
    row["normalized_distortion"] <= 0.05 && row["compute_proxy"] <= 0.75]
shortlist_pool = isempty(eligible) ? proxy_frontier : eligible
sort!(shortlist_pool; by = row -> (row["normalized_distortion"], row["compute_proxy"]))
shortlist = shortlist_pool[1:min(3, length(shortlist_pool))]

println("Shortlist (tối đa 3; quyết định sơ bộ):")
for row in shortlist
    println(row["config_id"], " | proxy=", round(row["compute_proxy"]; digits = 4),
        " | normalized distortion=", round(row["normalized_distortion"]; digits = 6),
        " | Spearman=", round(row["distance_rank_spearman"]; digits = 4),
        " | prep_s=", round(row["prep_time_s"]; digits = 2),
        " | unique=", row["n_unique"], "/", N)
end

SUMMARY_PATH = TC.write_summary(joinpath(REPORT_DIR, "tc4_vector_tradeoff_60k_summary.md"),
    shortlist, N, state.data[:tau], state.data[:patch_duplicate_summary]["train"].duplicate_rate,
    PCA_FIT_SECONDS, length(proxy_frontier))
println("Summary: ", SUMMARY_PATH)

## 12. Ghi metadata và kiểm tra output

Metadata ghi checksum của cache descriptor, config, số descriptor thực tế, seed cặp Spearman và output. Kiểm tra lại rằng các output đều hiện diện; notebook kết thúc tại mức vector, chưa sửa pipeline TC4.

In [ ]:
metadata = Dict(
    "experiment" => "tc4_vector_tradeoff_60k",
    "phase" => "pre_kmeans_only",
    "run_kmeans" => false,
    "train_images" => length(state.data[:train_ids]),
    "n_descriptors" => N,
    "raw_dimension" => 16,
    "tau_train" => state.data[:tau],
    "pca_fit_seconds" => PCA_FIT_SECONDS,
    "pair_sample_size" => PAIR_SAMPLE_SIZE,
    "pair_seed" => PAIR_SEED,
    "descriptor_sha256" => state.data[:descriptor_checksums]["train"],
    "archive_sha256" => state.data[:archive_checksums],
    "raw_duplicate_rate" => state.data[:patch_duplicate_summary]["train"].duplicate_rate,
    "julia_version" => string(VERSION),
    "config_path" => relpath(CONFIG_PATH, REPO_ROOT),
    "results_csv" => basename(CSV_PATH),
    "summary_md" => basename(SUMMARY_PATH),
    "figures" => [basename(FIGURE_PROXY), basename(FIGURE_PREP)],
)
open(joinpath(REPORT_DIR, "metadata.toml"), "w") do io
    TOML.print(io, metadata; sorted = true)
end

outputs = [CSV_PATH, SUMMARY_PATH, FIGURE_PROXY, FIGURE_PREP,
    joinpath(REPORT_DIR, "metadata.toml")]
output_status = [(path = basename(path), exists = isfile(path), bytes = isfile(path) ? filesize(path) : 0)
    for path in outputs]
@assert all(row -> row.exists && row.bytes > 0, output_status)
foreach(println, output_status)


## Cách diễn giải trước khi chuyển sang K-means

- **Compute proxy thấp**: ít phép tính khoảng cách ước tính hơn nếu K và số iteration giữ nguyên; không đảm bảo chạy nhanh hơn trên phần cứng thực.
- **Normalized distortion thấp**: tái tạo vector gần raw hơn so với phương sai tự nhiên trung bình.
- **Spearman cao**: thứ tự khoảng cách của các cặp mẫu được giữ tương đối tốt; đây không phải bằng chứng cụm tốt.
- **Weights tập trung**: một số key có count lớn có thể làm Weighted K-means bị chi phối; xem `max_weight`, P95 và tỷ lệ singleton.
- **Giới hạn**: các metric pha A chỉ là sàng lọc vector. Pha B bên dưới đo runtime Weighted K-means K=16 và Raw-space Validation Distortion; kết quả vẫn chưa đánh giá visual-word transactions. Thử nghiệm K=64 là bước riêng.

# Pha B — Benchmark Weighted K-means K=16

Pha này dùng biểu diễn đã khảo sát ở pha A nhưng dựng lại trực tiếp từ `X`; CSV tổng hợp không được dùng để suy diễn hay thay thế vector. Mỗi cấu hình được dựng một lần, sau đó fit 3 lần với seed 42. Kết quả được lưu riêng tại `results/fashion/tc/tc4_weighted_kmeans_k16/`.

Weighted K-means tối thiểu hóa tổng bình phương khoảng cách có nhân `weight`. Cả khởi tạo k-means++ lẫn cập nhật centroid đều dùng weight. Không nhân bản dữ liệu theo weight.

## B1. Chọn cấu hình và cố định điều kiện benchmark

Chọn một tập con không rỗng, không lặp của 14 ID từ pha A. `B0` được gợi ý để có thể tính speedup; nếu bỏ B0, cột speedup sẽ để trống. BLAS được cố định một luồng, còn số luồng Julia đang chạy được ghi metadata.

In [229]:
const K = 256
const SEED = 42
SELECTED_CONFIGS =  [
    "B0",
    "P12", "P08", "P04",
    "Q05", "Q10", "Q20", "Q32",
    "PQ12_A10", "PQ12_A25", "PQ12_A50",
    "PQ08_A10", "PQ08_A25", "PQ08_A50"
]
const N_REPEATS = 3
const N_INIT = 1
const MAX_ITER = 1000
const TOL = 1e-4
const BENCHMARK_BLAS_THREADS = 1
const WEIGHTED_REPORT_DIR = joinpath(REPO_ROOT, "results", "fashion", "tc", """tc4_weighted_kmeans_k$(K)""")
const NOTEBOOK_PATH = joinpath(REPO_ROOT, "notebook", "UD2", "experiments",
    "04_tc4_vector_pre_kmeans_tradeoff_60k.ipynb")

mkpath(WEIGHTED_REPORT_DIR)
BLAS.set_num_threads(BENCHMARK_BLAS_THREADS)
println("Selected configs: ", SELECTED_CONFIGS)
println("K=$K, seed=$SEED, repeats=$N_REPEATS, max_iter=$MAX_ITER, tol=$TOL")
println("Julia threads: ", Threads.nthreads(), "; BLAS threads: ", BLAS.get_num_threads())


Selected configs: ["B0", "P12", "P08", "P04", "Q05", "Q10", "Q20", "Q32", "PQ12_A10", "PQ12_A25", "PQ12_A50", "PQ08_A10", "PQ08_A25", "PQ08_A50"]
K=256, seed=42, repeats=3, max_iter=1000, tol=0.0001
Julia threads: 1; BLAS threads: 1


## B2. Xác minh nguồn và danh sách cấu hình

Kiểm tra metadata của pha A, checksum descriptor train, số descriptor và cờ `run_kmeans=false`. Mỗi cấu hình phải có trong kết quả vừa tính, qua sanity checks, có ít nhất K vector duy nhất và không trùng ID trong danh sách chọn.

In [230]:
using SHA

const VALID_CONFIG_IDS = Set( [
    "B0",
    "P12", "P08", "P04",
    "Q05", "Q10", "Q20", "Q32",
    "PQ12_A10", "PQ12_A25", "PQ12_A50",
    "PQ08_A10", "PQ08_A25", "PQ08_A50"
])
source_metadata_path = joinpath(REPORT_DIR, "metadata.toml")
source_metadata = TOML.parsefile(source_metadata_path)
source_rows = Dict(row["config_id"] => row for row in results)

source_metadata["experiment"] == "tc4_vector_tradeoff_60k" ||
    error("Metadata không thuộc thí nghiệm tc4_vector_tradeoff_60k")
source_metadata["phase"] == "pre_kmeans_only" ||
    error("Metadata nguồn không phải kết quả trước K-means")
source_metadata["run_kmeans"] == false ||
    error("Kết quả nguồn TC4 không hợp lệ: pha vector phải chưa chạy K-means")
source_metadata["n_descriptors"] == N ||
    error("Số descriptor trong metadata không khớp N hiện tại")
source_metadata["descriptor_sha256"] == state.data[:descriptor_checksums]["train"] ||
    error("Checksum descriptor train không khớp metadata TC4")
isfile(CSV_PATH) || error("Không tìm thấy CSV metric TC4: $CSV_PATH")

isempty(SELECTED_CONFIGS) && throw(ArgumentError("SELECTED_CONFIGS phải có ít nhất một config_id"))
length(unique(SELECTED_CONFIGS)) == length(SELECTED_CONFIGS) ||
    throw(ArgumentError("SELECTED_CONFIGS có config_id bị lặp"))
invalid_ids = setdiff(SELECTED_CONFIGS, collect(VALID_CONFIG_IDS))
isempty(invalid_ids) || throw(ArgumentError(
    "config_id không hợp lệ: $(join(invalid_ids, ", ")). Kiểm tra danh sách 14 ID trong hướng dẫn."))

for config_id in SELECTED_CONFIGS
    haskey(source_rows, config_id) || error("Không có config $config_id trong kết quả TC4")
    row = source_rows[config_id]
    row["passes_sanity_checks"] === true || error("Config $config_id không qua sanity checks TC4")
    row["n_unique"] >= K || error("Config $config_id có ít hơn K=$K vector duy nhất")
    row["n_input"] == N || error("N của $config_id không khớp dữ liệu train hiện tại")
end

println("Nguồn TC4 hợp lệ; descriptor SHA-256: ", source_metadata["descriptor_sha256"])
println("Sẽ benchmark ", length(SELECTED_CONFIGS), " cấu hình: ", join(SELECTED_CONFIGS, ", "))


Nguồn TC4 hợp lệ; descriptor SHA-256: 8ad47768b32da2f94c3ccf9ff8a6a1d010eef41daf5173b3df137d49895a63a2
Sẽ benchmark 14 cấu hình: B0, P12, P08, P04, Q05, Q10, Q20, Q32, PQ12_A10, PQ12_A25, PQ12_A50, PQ08_A10, PQ08_A25, PQ08_A50


## B3. Dựng biểu diễn Z và weights theo đúng cấu hình TC4

B0 dùng raw 16D; P dùng PCA score; Q dùng pixel đã lượng tử hóa rồi dedup; PQ dùng PCA score đã lượng tử hóa rồi dedup. Với Q/PQ, các khóa được sắp xếp xác định trước khi dựng Z, weights là count tương ứng. Thời gian prep gồm dựng biểu diễn; với PCA, cộng thêm thời gian fit basis dùng chung ở pha A.

In [ ]:
println("Các cấu hình biểu diễn đã chọn: ", join(SELECTED_CONFIGS, ", "))

## B4. Weighted K-means++ và thuật toán fit

Khởi tạo tâm đầu theo xác suất weight; các tâm sau theo `weight × D²`. Mỗi vòng gán vector vào tâm gần nhất, cập nhật centroid bằng trung bình có trọng số. Cụm rỗng được tái khởi tạo bằng vector có weighted error lớn nhất chưa dùng làm điểm cứu; hòa điểm chọn chỉ số nhỏ nhất. Dừng khi thay đổi tương đối tuyệt đối của weighted objective giữa hai vòng liên tiếp ≤ TOL.

In [ ]:
println("Weighted K-means: ", parentmodule(TC.weighted_kmeans), ".weighted_kmeans")

## B5. Unit test toy: weighted data tương đương dữ liệu nhân bản

Trên toy nhỏ, dùng các tâm khởi tạo cố định để loại khác biệt do ngẫu nhiên. So sánh centroid, assignment và inertia giữa dữ liệu nén có weights với dữ liệu lặp theo weights. Việc `repeat` chỉ dùng ở toy, không áp dụng cho Fashion-MNIST.

In [ ]:
Z_toy = Float64[0 2 10; 0 2 10]
weights_toy = [3, 2, 1]
initial_centers_toy = Float64[0 10; 0 10]
weighted_toy = TC.weighted_kmeans(Z_toy, weights_toy; k = 2, max_iter = 20,
    tolerance = 1e-12, initial_centers = initial_centers_toy)
expanded_toy = hcat([repeat(Z_toy[:, index:index], 1, weights_toy[index])
    for index in eachindex(weights_toy)]...)
expanded_fit_toy = TC.weighted_kmeans(expanded_toy, ones(Int, size(expanded_toy, 2));
    k = 2, max_iter = 20, tolerance = 1e-12, initial_centers = initial_centers_toy)

@assert isapprox(weighted_toy.centroids, expanded_fit_toy.centroids; atol = 1e-10)
@assert isapprox(weighted_toy.weighted_inertia, expanded_fit_toy.weighted_inertia; atol = 1e-10)
for index in eachindex(weights_toy)
    @assert count(==(weighted_toy.labels[index]), expanded_fit_toy.labels) ==
        sum(weights_toy[j] for j in eachindex(weights_toy)
            if weighted_toy.labels[j] == weighted_toy.labels[index])
end
println("Toy weighted-vs-expanded passed; centroid = ", weighted_toy.centroids)
println("Weighted inertia = ", weighted_toy.weighted_inertia)

## B6. Warm-up và benchmark 3 lần mỗi cấu hình

Warm-up dùng ma trận nhỏ cùng dtype và cùng implementation, không được tính vào số đo. Trước mỗi repeat gọi GC ngoài vùng đo; timer bao gồm weighted k-means++/khởi tạo, assignment, cập nhật, xử lý cụm rỗng và kiểm tra hội tụ.

In [ ]:
@assert N_INIT == 1 "Implementation notebook hiện cố định một lần khởi tạo mỗi repeat."

warmup_rng = MersenneTwister(SEED)
Z_warmup = rand(warmup_rng, Float32, 16, 512)
weights_warmup = ones(Int, size(Z_warmup, 2))
TC.weighted_kmeans(Z_warmup, weights_warmup; k = K, seed = SEED,
    max_iter = min(MAX_ITER, 5), tolerance = TOL)
GC.gc()
println("Warm-up hoàn tất; lượt này không được tính vào benchmark.")

In [ ]:
fit_config = (; K, SEED, MAX_ITER, TOL, N_REPEATS)
benchmark_results = NamedTuple[]
benchmark_run_results = NamedTuple[]
all_runs = NamedTuple[]
for config_id in SELECTED_CONFIGS
    config_result = TC.benchmark_one_config(config_id, source_rows[config_id], features, fit_config)
    push!(benchmark_results, (; config_id = config_result.config_id,
        method = config_result.method, k = config_result.k, seed = config_result.seed,
        n_input = config_result.n_input, n_unique = config_result.n_unique,
        d_eff = config_result.d_eff, sum_weights = config_result.sum_weights,
        prep_time_s = config_result.prep_time_s,
        fit_time_min_s = config_result.fit_time_min_s,
        fit_time_median_s = config_result.fit_time_median_s,
        fit_time_mean_s = config_result.fit_time_mean_s,
        fit_time_std_s = config_result.fit_time_std_s,
        total_time_median_s = config_result.total_time_median_s,
        n_iter = config_result.n_iter, converged = config_result.converged,
        weighted_inertia = config_result.weighted_inertia,
        empty_cluster_count = config_result.empty_cluster_count,
        passes_sanity_checks = config_result.passes_sanity_checks))
    append!(all_runs, config_result.runs)
    push!(benchmark_run_results, config_result)
    println("Hoàn tất ", config_id)
end

## B6.1. Centroid MSE so với codebook baseline

Đưa centroid của mọi cấu hình về Raw 16D, rồi dùng phép ghép cặp tối ưu Hungarian để bỏ qua việc ID cụm bị hoán vị. Với B0 làm tham chiếu:

$$
\mathrm{CentroidMSE}=\frac{1}{16K}\min_{\pi}\sum_{k=1}^{K}\left\|c_k^{\mathrm{B0}}-c_{\pi(k)}^{\mathrm{candidate}}\right\|_2^2
$$

MSE thấp nghĩa là codebook candidate gần centroid B0 hơn; đây là độ lệch so với baseline, không phải thước đo chất lượng trên dữ liệu validation. Notebook cũng báo MSE giữa từng repeat và repeat đại diện của chính cấu hình. Vì mọi repeat đều dùng seed 42, chỉ số này kiểm tra tính lặp lại dưới cùng seed, không đo độ nhạy theo seed. Cả hai metric đều tính sau khi fit nên không làm tăng `fit_time_s`.

In [ ]:
has_baseline_for_centroids = any(result -> result.config_id == "B0", benchmark_run_results)
baseline_raw_centroids = if has_baseline_for_centroids
    baseline_config = only(filter(result -> result.config_id == "B0", benchmark_run_results))
    TC.centroids_to_raw_space("B0", TC.representative_run(baseline_config).centroids, pca)
else
    nothing
end

all_runs = NamedTuple[]
centroid_summary_by_config = Dict{String,NamedTuple}()
for config_result in benchmark_run_results
    config_reference = TC.centroids_to_raw_space(config_result.config_id,
        TC.representative_run(config_result).centroids, pca)
    config_mse_to_baseline = Float64[]
    config_repeat_mse = Float64[]
    for run in config_result.runs
        candidate_raw_centroids = TC.centroids_to_raw_space(config_result.config_id, run.centroids, pca)
        mse_to_baseline = has_baseline_for_centroids ?
            TC.centroid_mse(baseline_raw_centroids, candidate_raw_centroids) : missing
        mse_within_config = TC.centroid_mse(config_reference, candidate_raw_centroids)
        !ismissing(mse_to_baseline) && push!(config_mse_to_baseline, mse_to_baseline)
        push!(config_repeat_mse, mse_within_config)
        push!(all_runs, merge(
            (; config_id = run.config_id, repeat = run.repeat, fit_time_s = run.fit_time_s,
                n_iter = run.n_iter, converged = run.converged,
                weighted_inertia = run.weighted_inertia,
                empty_cluster_count = run.empty_cluster_count),
            (; centroid_mse_to_b0_raw = mse_to_baseline,
                centroid_mse_within_config_raw = mse_within_config,
                centroids_fit_space = TC.encode_centroids(run.centroids))))
    end
    centroid_summary_by_config[config_result.config_id] = (
        centroid_mse_to_b0_raw_median = isempty(config_mse_to_baseline) ? missing : median(config_mse_to_baseline),
        centroid_mse_to_b0_raw_std = isempty(config_mse_to_baseline) ? missing : std(config_mse_to_baseline; corrected = false),
        centroid_mse_within_config_raw_median = median(config_repeat_mse),
        centroid_mse_within_config_raw_max = maximum(config_repeat_mse),
    )
end

benchmark_results = [merge(result, centroid_summary_by_config[result.config_id])
    for result in benchmark_results]
println("Centroid MSE đã tính sau khi fit; baseline có mặt: ", has_baseline_for_centroids)

## B6.2. Raw-space Validation Distortion

Đánh giá mỗi codebook trên các Raw Pixel descriptor 16D từ split validation (10.000 ảnh gốc test trong quy ước project). Với từng descriptor validation, tìm centroid Raw 16D gần nhất; lấy trung bình bình phương khoảng cách trên mọi chiều và descriptor:

\[
\mathrm{RawSpaceValidationDistortion}=\frac{1}{16N_{val}}\sum_{i=1}^{N_{val}}\min_{k=1}^{K}\left\|x_i^{val}-c_k^{raw}\right\|_2^2.
\]

Giá trị thấp nghĩa là codebook biểu diễn descriptor validation gần hơn. Validation chỉ được dùng ở bước chấm điểm này: ngưỡng Q10, PCA basis, vector lượng tử hóa và centroid đều fit từ train. Chỉ số này khác Centroid MSE: một metric đo sai khác codebook so với B0, metric kia đo độ phủ descriptor validation.

In [ ]:
X_validation = state.data[:validation_desc]
N_validation = size(X_validation, 2)
validation_descriptor_sha256 = state.data[:descriptor_checksums]["validation"]

@assert length(state.data[:validation_ids]) == 10_000
@assert size(X_validation, 1) == 16
@assert N_validation > 0
@assert N_validation == length(state.data[:validation_map])
@assert all(isfinite, X_validation)
@assert all(value -> 0f0 <= value <= 1f0, X_validation)

validation_distortion_by_run = Dict{Tuple{String,Int},Float64}()
validation_distortion_cache = Dict{String,Float64}()
validation_values_by_config = Dict{String,Vector{Float64}}()
for config_result in benchmark_run_results
    values = Float64[]
    for run in config_result.runs
        raw_centroids = TC.centroids_to_raw_space(config_result.config_id, run.centroids, pca)
        fingerprint = TC.centroid_fingerprint(raw_centroids)
        distortion = get!(validation_distortion_cache, fingerprint) do
            TC.raw_space_validation_distortion(X_validation, raw_centroids)
        end
        validation_distortion_by_run[(config_result.config_id, run.repeat)] = distortion
        push!(values, distortion)
    end
    validation_values_by_config[config_result.config_id] = values
end

all_runs = [merge(run, (; raw_space_validation_distortion =
    validation_distortion_by_run[(run.config_id, run.repeat)])) for run in all_runs]
validation_summary_by_config = Dict(config_id => (
    raw_space_validation_distortion_median = median(values),
    raw_space_validation_distortion_std = std(values; corrected = false),
) for (config_id, values) in validation_values_by_config)
benchmark_results = [merge(result, validation_summary_by_config[result.config_id])
    for result in benchmark_results]
println("Validation descriptors: ", N_validation)
println("Validation descriptor checksum: ", validation_descriptor_sha256)
println("Validation distortion cached fits: ", length(validation_distortion_cache),
    " distinct codebook(s) across ", length(benchmark_run_results) * N_REPEATS, " repeats")

## B7. Tính speedup thực đo và xuất CSV

`speedup_fit` và `speedup_total` chỉ được tính khi B0 nằm trong chính danh sách đã chạy. Các trường `n_iter`, `converged`, inertia và số cụm rỗng là của repeat có thời gian gần median nhất; `runs.csv` giữ đầy đủ từng repeat. Summary CSV có median/std Centroid MSE so với B0 và MSE giữa repeats. `runs.csv` có thêm cột `centroids_fit_space`: mỗi centroid ngăn bằng `|`, các tọa độ trong centroid ngăn bằng `;`; thứ tự centroid theo cột của ma trận Julia.

In [ ]:
has_baseline = any(row -> row.config_id == "B0", benchmark_results)
baseline_result = has_baseline ? only(filter(row -> row.config_id == "B0", benchmark_results)) : nothing
summary_rows = NamedTuple[]
for row in benchmark_results
    speedup_fit = has_baseline ? baseline_result.fit_time_median_s / row.fit_time_median_s : missing
    speedup_total = has_baseline ? baseline_result.total_time_median_s / row.total_time_median_s : missing
    push!(summary_rows, (; config_id = row.config_id, method = row.method, k = row.k,
        seed = row.seed, n_input = row.n_input, n_unique = row.n_unique, d_eff = row.d_eff,
        sum_weights = row.sum_weights, prep_time_s = row.prep_time_s,
        fit_time_min_s = row.fit_time_min_s, fit_time_median_s = row.fit_time_median_s,
        fit_time_mean_s = row.fit_time_mean_s, fit_time_std_s = row.fit_time_std_s,
        total_time_median_s = row.total_time_median_s, speedup_fit, speedup_total,
        n_iter = row.n_iter, converged = row.converged,
        weighted_inertia = row.weighted_inertia,
        empty_cluster_count = row.empty_cluster_count,
        centroid_mse_to_b0_raw_median = row.centroid_mse_to_b0_raw_median,
        centroid_mse_to_b0_raw_std = row.centroid_mse_to_b0_raw_std,
        centroid_mse_within_config_raw_median = row.centroid_mse_within_config_raw_median,
        centroid_mse_within_config_raw_max = row.centroid_mse_within_config_raw_max,
        raw_space_validation_distortion_median = row.raw_space_validation_distortion_median,
        raw_space_validation_distortion_std = row.raw_space_validation_distortion_std,
        passes_sanity_checks = row.passes_sanity_checks))
end

if has_baseline
    baseline_validation_distortion = baseline_result.raw_space_validation_distortion_median
    summary_rows = [merge(row, (; raw_space_validation_distortion_ratio_to_b0 =
        row.raw_space_validation_distortion_median / baseline_validation_distortion))
        for row in summary_rows]
else
    summary_rows = [merge(row, (; raw_space_validation_distortion_ratio_to_b0 = missing))
        for row in summary_rows]
end

summary_header = ["config_id", "method", "k", "seed", "n_input", "n_unique", "d_eff",
    "sum_weights", "prep_time_s", "fit_time_min_s", "fit_time_median_s",
    "fit_time_mean_s", "fit_time_std_s", "total_time_median_s", "speedup_fit",
    "speedup_total", "n_iter", "converged", "weighted_inertia", "empty_cluster_count",
    "centroid_mse_to_b0_raw_median", "centroid_mse_to_b0_raw_std",
    "centroid_mse_within_config_raw_median", "centroid_mse_within_config_raw_max",
    "raw_space_validation_distortion_median", "raw_space_validation_distortion_std",
    "raw_space_validation_distortion_ratio_to_b0", "passes_sanity_checks"]
runs_header = ["config_id", "repeat", "fit_time_s", "n_iter", "converged",
    "weighted_inertia", "empty_cluster_count", "centroid_mse_to_b0_raw",
    "centroid_mse_within_config_raw", "raw_space_validation_distortion",
    "centroids_fit_space"]
SUMMARY_CSV_PATH = TC.benchmark_csv(joinpath(WEIGHTED_REPORT_DIR,
    "weighted_kmeans_k$(K)_timing_60k.csv"), summary_header, summary_rows)
RUNS_CSV_PATH = TC.benchmark_csv(joinpath(WEIGHTED_REPORT_DIR, "runs_$(K).csv"), runs_header, all_runs)
println("Summary CSV: ", SUMMARY_CSV_PATH)
println("Repeat CSV: ", RUNS_CSV_PATH)

## B8. Runtime chart và so sánh với compute proxy

Biểu đồ dùng median fit/total thực đo và thêm đồ thị thời gian so với centroid MSE. Điểm gần đáy trái nghĩa là fit nhanh và codebook gần baseline hơn. Centroid MSE đo độ khác biệt codebook; Raw-space Validation Distortion đo khả năng biểu diễn descriptors validation bằng codebook. Validation chỉ dùng ở bước chấm điểm, không tham gia huấn luyện. Correlation xếp hạng compute proxy với fit runtime chỉ có ý nghĩa trên các cấu hình đã chọn.

In [ ]:
if has_baseline || length(summary_rows) >= 2
    sort!(summary_rows; by = row -> row.fit_time_median_s)
    FIT_CHART = TC.timing_bar_svg(summary_rows, :fit_time_median_s,
        "Weighted K-means K=16 — median fit runtime",
        joinpath(WEIGHTED_REPORT_DIR, "median_fit_time.svg"))
    sort!(summary_rows; by = row -> row.total_time_median_s)
    TOTAL_CHART = TC.timing_bar_svg(summary_rows, :total_time_median_s,
        "Weighted K-means K=16 — median total runtime",
        joinpath(WEIGHTED_REPORT_DIR, "median_total_time.svg"))
else
    FIT_CHART = ""
    TOTAL_CHART = ""
end

if has_baseline_for_centroids && length(summary_rows) >= 2

    CENTROID_TRADEOFF_CHART = TC.centroid_time_tradeoff_svg(summary_rows,
        joinpath(WEIGHTED_REPORT_DIR, "fit_time_vs_centroid_mse_b0.svg"))
else
    CENTROID_TRADEOFF_CHART = ""
    println("Centroid MSE so với B0/đồ thị tradeoff không có vì chưa chọn B0.")
end

if length(summary_rows) >= 2

    VALIDATION_TRADEOFF_CHART = TC.validation_distortion_tradeoff_svg(summary_rows,
        joinpath(WEIGHTED_REPORT_DIR, "fit_time_vs_validation_distortion.svg"))
else
    VALIDATION_TRADEOFF_CHART = ""
end

selected_proxy = [source_rows[row.config_id]["compute_proxy"] for row in summary_rows]
selected_runtime = [row.fit_time_median_s for row in summary_rows]
proxy_runtime_spearman = length(summary_rows) >= 2 ?
    TC.spearman_correlation(selected_proxy, selected_runtime) : missing
println("Spearman rank tương quan proxy/runtime: ", proxy_runtime_spearman)

## B9. Summary, metadata và kiểm tra bàn giao

Summary trả lời config nhanh nhất theo fit và end-to-end, speedup so với B0 nếu đã chạy, các config chưa hội tụ và mức tương quan compute proxy/runtime. Không kết luận chất lượng codebook chỉ từ thời gian hoặc inertia giữa các không gian khác chiều/thang đo.

In [ ]:
fastest_fit = argmin(row -> row.fit_time_median_s, summary_rows)
fastest_total = argmin(row -> row.total_time_median_s, summary_rows)
nonconverged = [row.config_id for row in summary_rows if !row.converged]
closest_to_baseline = has_baseline_for_centroids ?
    argmin(row -> row.centroid_mse_to_b0_raw_median, summary_rows) : nothing
most_repeat_stable = argmin(row -> row.centroid_mse_within_config_raw_max, summary_rows)
lowest_validation_distortion = argmin(
    row -> row.raw_space_validation_distortion_median, summary_rows)

summary_path = joinpath(WEIGHTED_REPORT_DIR, "summary.md")
open(summary_path, "w") do io
    println(io, "# Weighted K-means K=16 — benchmark 60k train")
    println(io)
    println(io, "- N descriptors: `", N, "`; K: `", K, "`; seed: `", SEED, "`.")
    println(io, "- Repeats/config: `", N_REPEATS, "`; max_iter: `", MAX_ITER,
        "`; relative objective tolerance: `", TOL, "`.")
    println(io, "- Nhanh nhất theo median fit: **", fastest_fit.config_id, "** (",
        round(fastest_fit.fit_time_median_s; digits = 3), " s).")
    println(io, "- Thấp nhất theo median total: **", fastest_total.config_id, "** (",
        round(fastest_total.total_time_median_s; digits = 3), " s).")
    println(io, "- Thấp nhất theo median Raw-space Validation Distortion: **",
        lowest_validation_distortion.config_id, "** (",
        round(lowest_validation_distortion.raw_space_validation_distortion_median; digits = 8), ").")
    if has_baseline_for_centroids
        println(io, "- Gần centroid B0 nhất theo median Centroid MSE Raw 16D: **",
            closest_to_baseline.config_id, "** (",
            round(closest_to_baseline.centroid_mse_to_b0_raw_median; digits = 8), ").")
    end
    println(io, "- Độ lệch lặp lại thấp nhất (median-to-repeat MSE tối đa): **",
        most_repeat_stable.config_id, "** (",
        round(most_repeat_stable.centroid_mse_within_config_raw_max; digits = 8), ").")
    if has_baseline
        println(io, "- Speedup fit và total so B0 được ghi trong CSV; B0 đã chạy cùng môi trường.")
    else
        println(io, "- Không có B0 trong lựa chọn; speedup để trống, không giả định baseline.")
    end
    println(io, "- Không hội tụ ở repeat đại diện: ", isempty(nonconverged) ? "không có" : join(nonconverged, ", "), ".")
    println(io, "- Spearman giữa compute proxy TC4 và median fit runtime trên lựa chọn này: ",
        ismissing(proxy_runtime_spearman) ? "không đủ cấu hình để tính" : string(round(proxy_runtime_spearman; digits = 4)), ".")
    println(io)
    println(io, "Raw-space Validation Distortion = tổng khoảng cách bình phương tới centroid gần nhất trên validation Raw 16D, chia cho 16 × số descriptor validation. Chỉ dùng validation để chấm điểm; không fit ngưỡng, PCA hay K-means trên split này.")
    println(io, "Centroid MSE so B0 đo độ khác biệt giữa codebook sau ghép cụm tối ưu; nó không thay cho Validation Distortion.")
    println(io, "`compute_proxy` không phải runtime. Inertia giữa các không gian vector khác chiều/thang đo không dùng làm so sánh chất lượng. Validation chỉ dùng để đánh giá distortion; không dùng để fit hay chọn ngưỡng/PCA/K-means. Benchmark không đánh giá visual vocabulary hoặc transactions.")
    println(io)
end

weighted_metadata = Dict(
    "experiment" => "tc4_weighted_kmeans_$(K)_timing_60k",
    "selected_configs" => SELECTED_CONFIGS,
    "k" => K,
    "seed" => SEED,
    "n_repeats" => N_REPEATS,
    "n_init" => N_INIT,
    "max_iter" => MAX_ITER,
    "tolerance" => TOL,
    "tolerance_rule" => "absolute relative change in weighted objective between consecutive iterations",
    "empty_cluster_policy" => "reinitialize to highest weighted-error vector; smallest index breaks ties",
    "source_experiment" => source_metadata["experiment"],
    "n_input" => N,
    "descriptor_sha256" => source_metadata["descriptor_sha256"],
    "tc4_csv_sha256" => bytes2hex(sha256(read(CSV_PATH))),
    "notebook_sha256" => isfile(NOTEBOOK_PATH) ? bytes2hex(sha256(read(NOTEBOOK_PATH))) : "unavailable",
    "git_commit" => TC.optional_git_commit(REPO_ROOT),
    "julia_version" => string(VERSION),
    "os" => string(Sys.KERNEL, " ", Sys.MACHINE),
    "cpu" => string(Sys.CPU_NAME),
    "ram_bytes" => Sys.total_memory(),
    "julia_threads" => Threads.nthreads(),
    "blas_threads" => BLAS.get_num_threads(),
    "numeric_type" => "Float32 input vectors; Float64 centroids/accumulation",
    "timing_method" => "time_ns; preparation and fit timed separately; GC before each repeat; warm-up excluded",
    "prep_includes_shared_pca_fit_seconds" => true,
    "centroid_mse_reference" => "Raw-space centroids matched to B0 with minimum-cost Hungarian assignment; divided by 16*K",
    "centroid_repeat_stability" => "Each repeat matched to the median-time representative centroids of the same config",
    "raw_space_validation_distortion_computed" => true,
    "validation_split" => "original_test used only for validation scoring",
    "validation_descriptor_count" => N_validation,
    "validation_descriptor_sha256" => validation_descriptor_sha256,
    "raw_space_validation_distortion_formula" => "sum_i min_k ||x_validation_i - centroid_raw_k||^2 / (16*N_validation)",
    "fit_includes" => "weighted k-means++ initialization, assignment, weighted updates, convergence and empty-cluster handling",
    "fit_excludes" => "data load, package load, output I/O, plotting, garbage collection",
)
open(joinpath(WEIGHTED_REPORT_DIR, "metadata.toml"), "w") do io
    TOML.print(io, weighted_metadata; sorted = true)
end

output_paths = [SUMMARY_CSV_PATH, RUNS_CSV_PATH, summary_path,
    joinpath(WEIGHTED_REPORT_DIR, "metadata.toml")]
!isempty(FIT_CHART) && push!(output_paths, FIT_CHART)
!isempty(TOTAL_CHART) && push!(output_paths, TOTAL_CHART)
!isempty(CENTROID_TRADEOFF_CHART) && push!(output_paths, CENTROID_TRADEOFF_CHART)
!isempty(VALIDATION_TRADEOFF_CHART) && push!(output_paths, VALIDATION_TRADEOFF_CHART)
@assert all(isfile, output_paths)
@assert all(row -> row.passes_sanity_checks && row.sum_weights == N, summary_rows)
println("Benchmark output:")
foreach(path -> println(relpath(path, REPO_ROOT), " (", filesize(path), " bytes)"), output_paths)

Centroid MSE Raw 16D trả lời: **codebook mới khác codebook gốc bao nhiêu?** MSE thấp nghĩa là gần B0 hơn, không tự chứng minh tốt hơn. MSE giữa repeats dùng cùng seed 42 nên kiểm tra tính lặp lại, không đo độ nhạy khi đổi seed. Raw-space Validation Distortion được báo riêng để đo sai số mô tả các descriptor validation.